# Calibration, sensor failures and latency (CS637)

Run top to bottom. Scripts live in `MyDrive/CS637/p3/extensions/`.

| Section | Needs | Runtime | Time |
|---|---|---|---|
| 2 Latency and model size | authors' checkpoints only | CPU is enough | ~3 min |
| 3 Sensor failures | raw PEMSD4 in `data/raw/` + Person 1's `process_pems.py` | T4 GPU | ~2 min data, ~3 min check, ~20-40 min full run |
| 4 Coverage + conformal | Person 1's prediction `.npz` files | any | a few min |

Switching runtime wipes `/content`: after a switch, run section 1 again (and the data cell in section 3).
All outputs go to `MyDrive/CS637/p3/`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
ROOT = '/content/drive/MyDrive/CS637'
P3 = f'{ROOT}/p3'
SCRIPTS = f'{P3}/extensions'
for d in (P3, SCRIPTS, f'{P3}/latency', f'{P3}/sensor_failure', f'{P3}/coverage'):
    os.makedirs(d, exist_ok=True)
print(sorted(os.listdir(SCRIPTS)))   # expect coverage_conformal.py, latency.py, p3_model_utils.py, sensor_failure.py

In [ ]:
# Our own copy of the three PEMSD4 Quantile variants (about 25 MB, with checkpoints).
# Cloned to /content so we never touch the shared uq4phy_traffic_network folder.
VARIANTS = ['quantile_model_PEMSD4', 'gpde_quantile_model_PEMSD4', 'gpde_quantile_phy_model_PEMSD4']
if not os.path.exists('/content/authors'):
    !git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/tianshubao1/uq4phy_traffic_network.git /content/authors
    !cd /content/authors && git sparse-checkout init --cone && git sparse-checkout set {' '.join(VARIANTS)} && git checkout -q
VDIRS = [f'/content/authors/{v}' for v in VARIANTS]
for v in VDIRS:
    print(v, sorted(os.listdir(f'{v}/models')))

## 2. Latency and model size (CPU)

The script hides the GPU, so this measures the CPU (note the CPU model in `latency_meta.json`). Defaults: batches 1/16/64, 10 timed passes each, about 3 minutes in total. Batch 256 (`--batches 1,16,64,256`) adds about 10 minutes per model on Colab's CPU and is not needed for the real-time argument.

`peak_rss_MB` in `model_size.csv` is the whole process (Python + PyTorch + the loader's batch-256 warm-up), not the model's memory; don't report it as such.

In [ ]:
args = ' '.join(f'--variant {v}' for v in VDIRS)
!python {SCRIPTS}/latency.py {args} --out {P3}/latency
import pandas as pd
display(pd.read_csv(f'{P3}/latency/model_size.csv'))
display(pd.read_csv(f'{P3}/latency/latency.csv'))

## 3. Sensor failures

Needs processed PEMSD4 (`train/val/test.npz`, channels [speed, occupancy, flow], 11871/1676/3376 samples). The next cell builds it on local disk with Person 1's `p1/scripts/process_pems.py` from `data/raw/pems04.npz` (writing large files to the shared Drive fails), then checks it.

In [ ]:
PEMSD4_DIR = '/content/data/PEMSD4'
!mkdir -p {PEMSD4_DIR}
!python {ROOT}/p1/scripts/process_pems.py --raw {ROOT}/data/raw/pems04.npz --out {PEMSD4_DIR}
!ls -la {PEMSD4_DIR}

import numpy as np
t = np.load(f'{PEMSD4_DIR}/test.npz')
print(t['x'].shape, t['y'].shape)     # expect (3376, 12, 307, 3) twice
print('mean speed', t['x'][..., 0].mean(), 'mean occupancy', t['x'][..., 1].mean())  # speed ~60 mph, occupancy < 1

In [ ]:
# Sanity check first: clean test MAE should be close to the authors' log at the saved epoch
#   quantile 2.6702 | gpde_quantile 2.1338 | gpde_quantile_phy 2.0189
# (small differences are expected: their evaluation also counts padded copies of the last sample)
!python {SCRIPTS}/sensor_failure.py --dataset_dir {PEMSD4_DIR} {args} --out {P3}/sensor_failure/clean_check --clean_only

In [ ]:
# Full experiment: dead and stuck sensors, 5/10/20/40 %, 3 random sensor sets, with and without the fix.
# Roughly 49 test-set passes per model; check the time of the first lines and reduce --seeds if needed.
!python {SCRIPTS}/sensor_failure.py --dataset_dir {PEMSD4_DIR} {args} --out {P3}/sensor_failure

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob(f'{P3}/sensor_failure/*.png')):
    print(f); display(Image(f))

## 4. Coverage and conformal calibration (from 22 Oct)

Uses Person 1's prediction files in `outputs/pemsd4/` (keys `val_pred`, `test_pred`, `val_truth`, `test_truth`). Run once on synthetic data first to see the script works.

In [ ]:
!python {SCRIPTS}/coverage_conformal.py --selftest --out /content/selftest | tail -3

In [ ]:
files = sorted(glob.glob(f'{ROOT}/outputs/pemsd4/*.npz'))
print(files)
npz_args = ' '.join(f'--npz {os.path.splitext(os.path.basename(f))[0]}={f}' for f in files)
!python {SCRIPTS}/coverage_conformal.py {npz_args} --out {P3}/coverage
for f in sorted(glob.glob(f'{P3}/coverage/*.png')):
    print(f); display(Image(f))